# Fine-tuning do BERTimbau — Emi YouTube Analytics

**Ensaio da Sprint 1.** Os rótulos são os da Gemini (`rotulo_fraco`): as métricas de
validação daqui dizem se o *pipeline* funciona, **não** quanto o modelo acerta. O
número do Capítulo 5 sai de `ml/avaliacao/`, contra o `rotulo_humano`, depois que o
gabarito voltar.

Este notebook **não reimplementa nada**: ele clona o repositório e chama os mesmos
módulos que rodam localmente (`ml.treino.treinar`, `ml.treino.exportar_onnx`). Código
duplicado entre notebook e repositório diverge na primeira correção, e aí o modelo
publicado deixa de ser o que o repositório descreve.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente → **GPU (T4)**.

In [ ]:
!nvidia-smi

## 1. Repositório e dependências

`ml/requirements-treino.txt` cobre tudo o que `ml/treino` importa: `asyncpg` (o banco),
`preprocessamento` (o `preparar_texto` que treino e inferência precisam executar
**identicamente** — CLAUDE.md Seção 3), `torch`, `transformers`, `onnx`, `onnxruntime`
e `psutil`.

Duas particularidades do Colab, e as duas já morderam uma vez:

- **o `preprocessamento` é reinstalado sem `-e`.** A instalação editável registra um
  `.pth` que o interpretador só lê ao iniciar: num kernel que já está rodando, ela só
  vale depois de *Reiniciar ambiente de execução*. A segunda linha troca a editável
  por uma instalação normal, que vale na hora;
- **a verificação no fim não é enfeite.** `!pip` que falha no Colab não interrompe o
  notebook: o erro rola para fora da tela e a célula seguinte quebra com um
  `ModuleNotFoundError` que não tem nada a ver com a causa. Foi assim que uma
  instalação incompleta apareceu como "No module named 'asyncpg'" três células
  adiante — o `asyncpg` está nos requisitos desde sempre; o que faltou foi a
  instalação inteira ter dado certo.

In [ ]:
!git clone https://github.com/emi-youtube/emi-youtube-analytics.git
%cd emi-youtube-analytics

!pip install -q -r ml/requirements-treino.txt
!pip install -q --force-reinstall --no-deps ./preprocessamento

### Verificação do ambiente

Importa tudo o que as próximas células vão usar. Se alguma coisa faltar, o erro
aparece **aqui**, com o nome do que faltou — e não três células adiante.

In [ ]:
import importlib

import preprocessamento
import torch

for modulo in (
    "asyncpg",
    "preprocessamento",
    "torch",
    "transformers",
    "onnx",
    "onnxruntime",
    "psutil",
    "numpy",
    "ml.treino.dados",
    "ml.treino.cartao",
    "ml.treino.treinar",
    "ml.treino.exportar_onnx",
    "ml.treino.prever_teste",
):
    importlib.import_module(modulo)

print("ambiente ok")
print("preprocessamento", preprocessamento.VERSAO)
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Credencial do banco

O `.env` **não está no repositório** (CLAUDE.md regra 1). Cole a `DATABASE_URL` no
cofre do Colab (🔑 no menu da esquerda) com o nome `DATABASE_URL`: assim ela não fica
escrita numa célula que o notebook salva junto com a saída.

In [ ]:
from pathlib import Path

from google.colab import userdata

Path(".env").write_text("DATABASE_URL=" + userdata.get("DATABASE_URL"), encoding="utf-8")
print(".env escrito (fora do repositorio: esta no .gitignore)")

## 3. Os dados

2.200 exemplos de `split IS NULL` com `rotulo_fraco`. Os 334 da amostra humana
(`split='teste'`) não entram — nem no treino, nem na validação.

A partição 85/15 é estratificada, com semente 42, e **existe só em memória**: nada é
gravado no banco. Se o Kappa ficar abaixo de 0,60, a nova amostra humana sai
justamente destes 2.200, e marcá-los aqui esvaziaria esse pool.

In [ ]:
import asyncio
import logging
import sys

from ml.treino.dados import (
    carregar_exemplos,
    dividir_estratificado,
    pesos_de_classe,
    relatar_particao,
)

logging.basicConfig(level="INFO", format="%(message)s", stream=sys.stdout, force=True)

ID_EXECUCAO = 4

exemplos = asyncio.run(carregar_exemplos(ID_EXECUCAO))
particao = dividir_estratificado(exemplos)
relatar_particao(particao, pesos_de_classe(particao.treino))

## 4. Busca de hiperparâmetros — pela validação, e só

Nove configurações (3 taxas de aprendizado × 3 números de época). Cada uma treina do
zero e é julgada pelo **F1 macro de validação**. O conjunto de teste não aparece em
nenhuma delas: olhar o teste e voltar para mexer na taxa de aprendizado transformaria
o teste num segundo conjunto de validação.

Na T4 são ~2 min por configuração — reserve uns 25 minutos. Para pular a busca e
treinar uma configuração só, vá direto para a célula 5.

In [ ]:
from ml.treino.treinar import (
    ARQUIVO_BUSCA,
    buscar_hiperparametros,
    escolher_dispositivo,
    gravar_busca,
)

dispositivo = escolher_dispositivo()
print("dispositivo:", dispositivo)

escolhido, resultados = buscar_hiperparametros(particao, dispositivo)
gravar_busca(resultados, escolhido, ARQUIVO_BUSCA)
print()
print("escolhido pela validacao:", escolhido)

## 5. Treino da configuração vencedora

Retreina com a mesma semente e guarda o melhor estado — "melhor" é a época de maior
F1 macro de **validação**, não a última.

In [ ]:
from pathlib import Path

from ml.treino.treinar import salvar_modelo, treinar_uma_vez

# Sem rodar a busca (celula 4), escolha na mao:
#   from ml.treino.treinar import Hiperparametros
#   escolhido = Hiperparametros(taxa_aprendizado=3e-5, epocas=3, lote=16)

resultado = treinar_uma_vez(particao, escolhido, dispositivo)
destino = salvar_modelo(resultado, particao, Path("ml/modelos/bertimbau-ensaio"))

print("melhor epoca:", resultado.melhor_epoca)
print("val F1 macro:", round(resultado.f1_macro, 4))

## 6. O `model_card.json`

O contrato com o `backend/`: `id2label` (a ordem dos rótulos sai daqui, nunca do
código), `max_length`, `versao_preprocessamento` (o worker recusa o modelo se
divergir da versão instalada), hiperparâmetros, semente e data.

In [ ]:
import json

cartao = json.loads((destino / "model_card.json").read_text(encoding="utf-8"))
print(json.dumps(cartao, ensure_ascii=False, indent=2))

## 7. ONNX int8 — o modelo cabe no Azure B1?

A conversão e a medição rodam em **CPU**, com **uma thread**, porque é isso que a
instância B1 tem (1 vCPU, 1,75 GB). Medir na T4 responderia a pergunta errada.

Cada formato é medido num **processo separado**: com os três no mesmo processo, o RSS
do ONNX sairia somado ao do BERT do PyTorch ainda carregado.

Portão: perder no máximo **1 ponto de F1 macro** em relação ao original.

In [ ]:
!python -m ml.treino.exportar_onnx --id-execucao 4 --modelo ml/modelos/bertimbau-ensaio

## 8. Levar os artefatos embora

O Colab apaga a máquina quando a sessão termina. Baixe a pasta do modelo (pesos,
tokenizer, `model_card.json` e os dois grafos ONNX) e os dois relatórios
versionáveis.

In [ ]:
from google.colab import files

!zip -qr bertimbau-ensaio.zip ml/modelos/bertimbau-ensaio
!cp ml/treino/busca_hiperparametros.json ml/treino/relatorio_onnx.json .

files.download("bertimbau-ensaio.zip")
files.download("busca_hiperparametros.json")
files.download("relatorio_onnx.json")

## 9. O conjunto de teste — uma única vez, no fim

`prever_teste.py` classifica os 334 **sem ler nenhum rótulo** e grava um CSV. Nenhuma
métrica sai dele: a comparação contra o gabarito humano é `ml/avaliacao/avaliar.py`,
um processo separado, rodado depois — e que só faz sentido quando o `rotulo_humano`
já existir.

Enquanto o gabarito não voltar, esta célula produz previsões que ficam esperando.

In [ ]:
!python -m ml.treino.prever_teste --id-execucao 4 --modelo ml/modelos/bertimbau-ensaio

# Depois que o gabarito voltar, localmente, com as previsoes em maos:
#   python -m ml.avaliacao.avaliar --id-execucao 4 --gemini
#     --previsoes bertimbau=ml/dados/previsoes_bertimbau.csv
#     --previsoes lexico=ml/dados/previsoes_lexico.csv